# 01 — Data Understanding: dividendos Trading 212

Objetivo: perceber a estrutura, qualidade e conteúdo do histórico de dividendos (`output/dividends.csv`) antes da limpeza e modelação.

## 1. Imports

In [202]:
import pandas as pd
from pathlib import Path

In [203]:
import yfinance as yf
from time import sleep

## 2. Carregamento dos dados

In [204]:
filename = 'dividends.csv'
file_path = Path.cwd().parent / 'output' / filename

In [205]:
df = pd.read_csv(file_path, sep=',')

## 3. Estrutura do dataset
Dimensão, tipos, colunas e primeiras linhas.

In [206]:
# Retorna o tamanho do dataframe/matriz
df.shape

(145, 11)

In [207]:
# Retorna as informações relevantes do dataframe
# O nome da coluna quantidade de dados não nulos e o tipo de dados
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 145 entries, 0 to 144
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   ticker               145 non-null    object 
 1   name                 145 non-null    object 
 2   isin                 145 non-null    object 
 3   currency             145 non-null    object 
 4   amountInEuro         145 non-null    float64
 5   paidOn               145 non-null    object 
 6   type                 145 non-null    object 
 7   grossAmountPerShare  145 non-null    float64
 8   reference            145 non-null    object 
 9   quantity             145 non-null    float64
 10  amount               145 non-null    float64
dtypes: float64(4), object(7)
memory usage: 12.6+ KB


In [208]:
# retorna todas as colunas do dataframe
df.columns

Index(['ticker', 'name', 'isin', 'currency', 'amountInEuro', 'paidOn', 'type',
       'grossAmountPerShare', 'reference', 'quantity', 'amount'],
      dtype='object')

In [209]:
df.head(5)

,ticker,name,isin,currency,amountInEuro,paidOn,type,grossAmountPerShare,reference,quantity,amount
0,SDIVm_EQ,Global X Superdividend (Dist),IE00077FRP95,EUR,0.58,2026-10-30T15:45:23.000+02:00,DIVIDEND,0.071554,4890ad14-254e-4878-b330-ed121f5ce758,8.126585,0.58
1,JEPIl_EQ,JPMorgan US Equity Premium Income Active (Dist),IE000U5MJOZ6,USD,0.04,2026-10-23T14:27:20.000+03:00,DIVIDEND,0.141100,8b78e8b4-8180-4c93-94cc-2c795b5de8bc,0.320549,0.04
2,JEPGl_EQ,JPMorgan Global Equity Premium Income Active (...,IE0003UVYC20,USD,0.41,2026-10-23T14:26:23.000+03:00,DIVIDEND,0.119300,03721238-a3b8-4d8a-8571-e9d9de4a34c8,3.829508,0.41
3,STAG_US_EQ,STAG Industrial,US85254J1025,USD,0.19,2026-10-15T18:12:58.000+03:00,DIVIDEND,0.387500,cc81d873-837e-4095-9bad-365af078ef65,0.631032,0.19
4,EPR_US_EQ,EPR Properties,US26884U1097,USD,1.12,2026-10-15T17:51:00.000+03:00,DIVIDEND,0.310000,631c89c0-0bca-4a9c-9ee1-83fe23a5e1b7,4.767652,1.12


## 4. Estatística descritiva

In [210]:
df.describe()

,amountInEuro,grossAmountPerShare,quantity,amount
count,145.000000,145.000000,145.000000,145.000000
mean,0.647172,0.255981,3.444938,0.647172
std,0.369276,0.106647,1.674770,0.369276
min,0.010000,0.070416,0.015431,0.010000
25%,0.260000,0.240000,2.195866,0.260000
50%,0.630000,0.265000,3.829508,0.630000
75%,0.940000,0.295000,4.667338,0.940000
max,1.570000,0.812120,8.126585,1.570000


In [211]:
df['amount'].median()

np.float64(0.63)

In [212]:
df['amountInEuro'].sort_values(ascending=True)

28     0.01
27     0.01
8      0.01
144    0.04
1      0.04
       ... 
87     1.46
73     1.50
58     1.56
20     1.56
40     1.57
Name: amountInEuro, Length: 145, dtype: float64

## 5. Enriquecimento: setor de cada ativo (yfinance)
Pesquisa por ISIN; ETFs ficam como `ETF`, ISINs sem resultado ficam como `Investigar`.

O mapa `isin → setor` fica guardado em `output/sectors.csv`. Se o ficheiro existir, a API não é chamada. Para voltar a buscar (ex.: entrou um ativo novo), apaga o ficheiro.

In [213]:
def get_sector(isin: str):

    sleep(0.5)
    dat = yf.Search(isin)
    quotes : list = dat.quotes

    if not quotes:
        return "Investigar"

    first_quote = quotes[0]
    quote_type = first_quote.get('quoteType', '')
    symbol = first_quote.get('symbol', '')

    if quote_type == 'ETF':
        return 'ETF'

    if symbol:
        info = yf.Ticker(symbol).info
        return info.get('sector', 'UNKNOW')
    
    return 'NOT INFORMED'

  

In [214]:
sector_path = Path.cwd().parent / 'output' / 'sectors.csv'

if sector_path.exists():
    mapa = pd.read_csv(sector_path).set_index('isin')['sector'].to_dict()
else:
    mapa = {
        isin : get_sector(isin) for isin in df['isin'].unique()
    }
    mapa['US6915438476'] = 'Financial Services'  # OXLC, confirmado manualmente, A API não retorna nada
    pd.Series(mapa, name='sector').rename_axis('isin').reset_index().to_csv(sector_path, index=False)

In [188]:
df['sector'] = df['isin'].map(mapa)

### 5.1 Investigação dos ISINs sem setor

In [174]:
teste = yf.Search('US6915438476')
display(teste.quotes)

[]

In [ ]:
# Pesquisa feita manualmente dá por conta que a oxford lane capital está no setor financeiro
df.iloc[142]

ticker                                           OXLC_US_EQ
name                                    Oxford Lane Capital
isin                                           US6915438476
currency                                                USD
amountInEuro                                           0.22
paidOn                        2024-08-30T14:09:47.000+03:00
type                                               DIVIDEND
grossAmountPerShare                                    0.09
reference              45daa557-5a76-4845-b87c-4d8e502c0747
quantity                                            3.18045
amount                                                 0.22
Name: 142, dtype: object

In [158]:
df['sector'] = df['sector'].replace('Investigar', 'Financial Services') 

### 5.2 Verificação final

In [156]:
df.columns

Index(['ticker', 'name', 'isin', 'currency', 'amountInEuro', 'paidOn', 'type',
       'grossAmountPerShare', 'reference', 'quantity', 'amount', 'sector'],
      dtype='object')

In [159]:
df['sector'].unique()

array(['ETF', 'Real Estate', 'Financial Services', 'Basic Materials',
       'Healthcare', 'Consumer Defensive'], dtype=object)